In [ ]:
import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import google.generativeai
import anthropic
from IPython.display import display, Markdown, update_display
import gradio as gr
import subprocess

load_dotenv()

openai = OpenAI()
claude = anthropic.Anthropic()

OPENAI_MODEL = "gpt-4o"
CLAUDE_MODEL = "claude-sonnet-4-20250514"

In [ ]:
system_message = "You are an assistant that reimplements Python code in high performance C++ for an M1 Mac. "
system_message += "Respond only with C++ code; use comments sparingly and do not provide any explanation other than occasional comments. "
system_message += "The C++ response needs to produce an identical output in the fastest possible time."

In [ ]:
def user_prompt_for(python):
    user_prompt = "Rewrite this Python code in C++ with the fastest possible implementation that produces identical output in the least time. "
    user_prompt += "Respond only with C++ code; do not explain your work other than a few comments. "
    user_prompt += "Pay attention to number types to ensure no int overflows. Remember to #include all necessary C++ packages such as iomanip.\n\n"
    user_prompt += python
    return user_prompt

In [ ]:
def messages_for(python):
    return [
        {'role': 'system', 'content': system_message},
        {'role': 'user', 'content': user_prompt_for(python)}
    ]

In [ ]:
def write_output(cpp):
    code = cpp.replace("```cpp", "").replace("```", "")
    with open("optimized.cpp", "w") as f:
        f.write(code)

In [ ]:
def optimize_gpt(python):
    stream = openai.chat.completions.create(
        model=OPENAI_MODEL,
        messages=messages_for(python),
        stream=True
    )

    reply = ""
    for chunk in stream:
        fragment = chunk.choices[0].delta.content or ""
        reply += fragment
        print(fragment, end="", flush=True)

    write_output(reply)

In [ ]:
def optimize_claude(python):
    result = claude.messages.stream(
        model = CLAUDE_MODEL,
        max_tokens = 2000,
        system = system_message,
        messages = [{"role": "user", "content": user_prompt_for(python)}]
    )

    reply = ""
    with result as stream:
        for text in stream.text_stream:
            reply += text
            print(text, end="", flush=True)
    write_output(reply)

In [ ]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(100_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [ ]:
exec(pi)

In [ ]:
optimize_gpt(pi)

In [ ]:
!clang++ -std=c++17 -O2 -o optimized optimized.cpp
!./optimized

In [ ]:
optimize_claude(pi)

In [ ]:
!clang++ -std=c++17 -O2 -o optimized optimized.cpp
!./optimized

In [ ]:
python_hard = """# Be careful to support large number sizes

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value
        
def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

In [ ]:
exec(python_hard)

In [ ]:
optimize_gpt(python_hard)

!clang++ -std=c++17 -O2 -o optimized optimized.cpp
!./optimized

In [ ]:
optimize_claude(python_hard)

!clang++ -std=c++17 -O2 -o optimized optimized.cpp
!./optimized

In [ ]:
def stream_gpt(python):    
    stream = openai.chat.completions.create(model=OPENAI_MODEL, messages=messages_for(python), stream=True)
    reply = ""
    for chunk in stream:
        fragment = chunk.choices[0].delta.content or ""
        reply += fragment
        yield reply.replace('```cpp\n','').replace('```','')



def stream_claude(python):
    result = claude.messages.stream(
        model=CLAUDE_MODEL,
        max_tokens=2000,
        system=system_message,
        messages=[{"role": "user", "content": user_prompt_for(python)}],
    )
    reply = ""
    with result as stream:
        for text in stream.text_stream:
            reply += text
            yield reply.replace('```cpp\n','').replace('```','')

In [ ]:
def optimize(python, model):
    if model=="GPT":
        result = stream_gpt(python)
    elif model=="Claude":
        result = stream_claude(python)
    else:
        raise ValueError("Unknown model")
    for stream_so_far in result:
        yield stream_so_far        

In [ ]:
# with gr.Blocks() as ui:
#     with gr.Row():
#         python = gr.Textbox(label="Python code:", lines=10, value=python_hard)
#         cpp = gr.Textbox(label="C++ code:", lines=10)
#     with gr.Row():
#         model = gr.Dropdown(["GPT", "Claude"], label="Select model", value="GPT")
#         convert = gr.Button("Convert code")

#     convert.click(optimize, inputs=[python, model], outputs=[cpp])

# ui.launch(inbrowser=True)

In [ ]:
def execute_python(code):
    try:
        output = io.StringIO()
        sys.stdout = output
        exec(code)
    finally:
        sys.stdout = sys.__stdout__
    return output.getvalue()


def execute_cpp(code):
    write_output(code)
    try:
        compile_cmd = ["clang++", "-std=c++17", "-O2", "-o", "optimized", "optimized.cpp"]
        compile_result = subprocess.run(compile_cmd, check=True, text=True, capture_output=True)
        run_cmd = ["/Users/slavacalestru/Desktop/DonerLLMEngineering_NF/New/week4/optimized"]
        run_result = subprocess.run(run_cmd, check=True, text=True, capture_output=True)
        return run_result.stdout
    except subprocess.CalledProcessError as e:
        return f"An error occurred:\n{e.stderr}"

In [ ]:
css = """
.python {background-color: #306998;}
.cpp {background-color: #050;}
"""

In [ ]:
# with gr.Blocks(css=css) as ui:
#     gr.Markdown("## Convert code from Python to C++")
#     with gr.Row():
#         python = gr.Textbox(label="Python code:", lines=10, value=python_hard)
#         cpp = gr.Textbox(label="C++ code:", lines=10)
#     with gr.Row():
#         model = gr.Dropdown(["GPT", "Claude"], label="Select model", value="GPT")
#     with gr.Row():
#         convert = gr.Button("Convert code")
#     with gr.Row():
#         python_run = gr.Button("Run Python code")
#         cpp_run = gr.Button("Run C++ code")
#     with gr.Row():
#         python_out = gr.TextArea(label="Python output", elem_classes=['python'])
#         cpp_out = gr.TextArea(label="C++ output", elem_classes=['cpp'])

#     convert.click(optimize, inputs=[python, model], outputs=[cpp])
#     python_run.click(execute_python, inputs=[python], outputs=[python_out])
#     cpp_run.click(execute_cpp, inputs=[cpp], outputs=[cpp_out])

# ui.launch(inbrowser=True)

In [ ]:
system_message_ds = "You are an assistant that given some Python code adds docstring and comments to the code."
system_message_ds += " Respond only with Python code; use comments sparingly and do not provide any explanation other than occasional comments."
system_message_ds += "Importantly, do not add any other information, explanation, text, or imports rather than the docstring and code comments."
system_message_ds += " Do not add ``` symbols to the code. Do not add the word 'python' at the beginning of the code."

In [ ]:
def user_prompt_ds(python):
    user_prompt = "Add docstring and comments to the following Python code. "
    user_prompt += "Respond only with Python code; do not explain your work other than a few comments. "
    user_prompt += "Pay attention to the code structure and do not change it. And ensure clarity in the comments.\n\n"
    user_prompt += python
    return user_prompt

In [ ]:
def messages_for_ds(python):
    return [
        {'role': 'system', 'content': system_message_ds},
        {'role': 'user', 'content': user_prompt_ds(python)}
    ]

In [ ]:
def stream_ds_gpt(python):    
    stream = openai.chat.completions.create(model=OPENAI_MODEL, messages=messages_for_ds(python), stream=True)
    reply = ""
    for chunk in stream:
        fragment = chunk.choices[0].delta.content or ""
        reply += fragment
        yield reply.replace('```cpp\n','').replace('```','')



def stream_ds_claude(python):
    result = claude.messages.stream(
        model=CLAUDE_MODEL,
        max_tokens=2000,
        system=system_message,
        messages=[{"role": "user", "content": user_prompt_ds(python)}],
    )
    reply = ""
    with result as stream:
        for text in stream.text_stream:
            reply += text
            yield reply.replace('```cpp\n','').replace('```','')


def generate_ds(python, model):
    if model=="GPT":
        result = stream_ds_gpt(python)
    elif model=="Claude":
        result = stream_ds_claude(python)
    else:
        raise ValueError("Unknown model")
    for stream_so_far in result:
        yield stream_so_far        

In [ ]:
# with gr.Blocks(css=css) as ui:
#     gr.Markdown("## Convert code from Python to C++")
#     with gr.Row():
#         python = gr.Textbox(label="Python code:", lines=10, value=pi)
#         cpp = gr.Textbox(label="C++ code:", lines=10)
#         python_ds = gr.Textbox(label="Python with docstring:", lines=10)
#     with gr.Row():
#         model = gr.Dropdown(["GPT", "Claude"], label="Select model", value="GPT")
#     with gr.Row():
#         convert = gr.Button("Convert code")
#         python_add_ds = gr.Button("Add docstrings")
#     with gr.Row():
#         python_run = gr.Button("Run Python code")
#         cpp_run = gr.Button("Run C++ code")
#     with gr.Row():
#         python_out = gr.TextArea(label="Python output", elem_classes=['python'])
#         cpp_out = gr.TextArea(label="C++ output", elem_classes=['cpp'])
#     with gr.Row():
#         tests_button = gr.Button("Generate Unit Tests")
#     with gr.Row():
#         tests_output = gr.TextArea(label="Unit Tests for python code")

#     convert.click(generate_response, inputs=[python, model, ], outputs=[cpp])
#     python_run.click(execute_python, inputs=[python], outputs=[python_out])
#     cpp_run.click(execute_cpp, inputs=[cpp], outputs=[cpp_out])
#     python_add_ds.click(generate_response, inputs=[python, model], outputs=[python_ds])

# ui.launch(inbrowser=True)

In [ ]:
system_message_tests = "You are an assistant that given some Python code will write unit test cases for the code."
system_message_tests += " Respond only with Python code; use comments sparingly and do not provide any explanation other than occasional comments."
system_message_tests += "Importantly, do not add any other information, explanation, text, or imports rather than the unit tests."
system_message_tests += " Do not add ``` symbols to the code."


def user_prompt_tests(python):
    user_prompt = "Write unit tests for the following Python code. "
    user_prompt += "Respond only with Python code; do not explain your work other than a few comments. "
    user_prompt += "Pay attention to the code structure and ensure clarity in the comments.\n\n"
    user_prompt += python
    return user_prompt


def messages_for_tests(python):
    return [
        {'role': 'system', 'content': system_message_tests},
        {'role': 'user', 'content': user_prompt_tests(python)}
    ]

In [ ]:



def stream_tests_gpt(python):    
    stream = openai.chat.completions.create(model=OPENAI_MODEL, messages=messages_for_tests(python), stream=True)
    reply = ""
    for chunk in stream:
        fragment = chunk.choices[0].delta.content or ""
        reply += fragment
        yield reply.replace('```cpp\n','').replace('```','')



def stream_tests_claude(python):
    result = claude.messages.stream(
        model=CLAUDE_MODEL,
        max_tokens=2000,
        system=system_message,
        messages=[{"role": "user", "content": user_prompt_tests(python)}],
    )
    reply = ""
    with result as stream:
        for text in stream.text_stream:
            reply += text
            yield reply.replace('```cpp\n','').replace('```','')


def generate_tests(python, model):
    if model=="GPT":
        result = stream_tests_gpt(python)
    elif model=="Claude":
        result = stream_tests_claude(python)
    else:
        raise ValueError("Unknown model")
    for stream_so_far in result:
        yield stream_so_far        

In [ ]:
with gr.Blocks(css=css) as ui:
    gr.Markdown("## Convert code from Python to C++")
    with gr.Row():
        python = gr.Textbox(label="Python code:", lines=10, value=pi)
        cpp = gr.Textbox(label="C++ code:", lines=10)
        python_ds = gr.Textbox(label="Python with docstring:", lines=10)
    with gr.Row():
        model = gr.Dropdown(["GPT", "Claude"], label="Select model", value="GPT")
    with gr.Row():
        convert = gr.Button("Convert code")
        python_add_ds = gr.Button("Add docstrings")
    with gr.Row():
        python_run = gr.Button("Run Python code")
        cpp_run = gr.Button("Run C++ code")
    with gr.Row():
        python_out = gr.TextArea(label="Python output", elem_classes=['python'])
        cpp_out = gr.TextArea(label="C++ output", elem_classes=['cpp'])
    with gr.Row():
        tests_button = gr.Button("Generate Unit Tests")
    with gr.Row():
        tests_out = gr.TextArea(label="Unit Tests for python code")

    convert.click(optimize, inputs=[python, model], outputs=[cpp])
    python_run.click(execute_python, inputs=[python], outputs=[python_out])
    cpp_run.click(execute_cpp, inputs=[cpp], outputs=[cpp_out])
    python_add_ds.click(generate_ds, inputs=[python, model], outputs=[python_ds])
    tests_button.click(generate_tests, inputs=[python, model], outputs=[tests_out])

ui.launch(inbrowser=True)